In [1]:
import os

# Correct path for SageMaker
save_path = os.path.expanduser("~/churn-mlops/src/serving/api.py")
os.makedirs(os.path.dirname(save_path), exist_ok=True)

api_code = '''
import os
import io
import json
import logging
import datetime
import boto3
import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn

# ── Logging setup ──────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("predictions.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ── Load model from S3 ─────────────────────────────────────
BUCKET_NAME = "churn-mlops-project-cherry"
s3 = boto3.client("s3")

def load_model():
    logger.info("Loading model from S3...")
    s3.download_file(
        BUCKET_NAME,
        "models/latest/xgboost_churn_model.pkl",
        "/tmp/model.pkl"
    )
    model = joblib.load("/tmp/model.pkl")
    logger.info("Model loaded successfully!")
    return model

def load_scaler():
    logger.info("Loading scaler from S3...")
    s3.download_file(
        BUCKET_NAME,
        "models/scaler.pkl",
        "/tmp/scaler.pkl"
    )
    scaler = joblib.load("/tmp/scaler.pkl")
    logger.info("Scaler loaded successfully!")
    return scaler

def load_metadata():
    obj = s3.get_object(Bucket=BUCKET_NAME, Key="models/latest/metadata.json")
    return json.loads(obj["Body"].read())

# Load everything when API starts
model    = load_model()
scaler   = load_scaler()
metadata = load_metadata()

# ── FastAPI App ────────────────────────────────────────────
app = FastAPI(
    title="Netflix Churn Prediction API",
    description="Predicts whether a Netflix customer will churn",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

# ── Input Schema ───────────────────────────────────────────
class CustomerData(BaseModel):
    age:                    float
    watch_hours:            float
    last_login_days:        float
    monthly_fee:            float
    number_of_profiles:     float
    avg_watch_time_per_day: float
    gender:                 str
    subscription_type:      str
    region:                 str
    payment_method:         str

# ── Helper: build feature row ──────────────────────────────
def build_features(data: CustomerData) -> pd.DataFrame:
    engagement_score   = (data.watch_hours * 0.5 +
                          data.avg_watch_time_per_day * 30 -
                          data.last_login_days * 0.3)
    is_low_engagement  = int(data.avg_watch_time_per_day < 0.5)
    is_long_inactive   = int(data.last_login_days > 30)
    fee_per_profile    = data.monthly_fee / data.number_of_profiles
    watch_to_fee_ratio = data.watch_hours / (data.monthly_fee + 1)

    if data.age < 25:
        age_group = "young"
    elif data.age < 40:
        age_group = "adult"
    elif data.age < 55:
        age_group = "middle_aged"
    else:
        age_group = "senior"

    row = {
        "age":                        data.age,
        "watch_hours":                data.watch_hours,
        "last_login_days":            data.last_login_days,
        "monthly_fee":                data.monthly_fee,
        "number_of_profiles":         data.number_of_profiles,
        "avg_watch_time_per_day":     data.avg_watch_time_per_day,
        "engagement_score":           engagement_score,
        "is_low_engagement":          is_low_engagement,
        "is_long_inactive":           is_long_inactive,
        "fee_per_profile":            fee_per_profile,
        "watch_to_fee_ratio":         watch_to_fee_ratio,
        "gender_Female":              int(data.gender == "Female"),
        "gender_Male":                int(data.gender == "Male"),
        "gender_Other":               int(data.gender == "Other"),
        "subscription_type_Basic":    int(data.subscription_type == "Basic"),
        "subscription_type_Premium":  int(data.subscription_type == "Premium"),
        "subscription_type_Standard": int(data.subscription_type == "Standard"),
        "region_Africa":              int(data.region == "Africa"),
        "region_Asia":                int(data.region == "Asia"),
        "region_Europe":              int(data.region == "Europe"),
        "region_North America":       int(data.region == "North America"),
        "region_Oceania":             int(data.region == "Oceania"),
        "region_South America":       int(data.region == "South America"),
        "payment_method_Credit Card": int(data.payment_method == "Credit Card"),
        "payment_method_Crypto":      int(data.payment_method == "Crypto"),
        "payment_method_Debit Card":  int(data.payment_method == "Debit Card"),
        "payment_method_Gift Card":   int(data.payment_method == "Gift Card"),
        "payment_method_PayPal":      int(data.payment_method == "PayPal"),
        "age_group_adult":            int(age_group == "adult"),
        "age_group_middle_aged":      int(age_group == "middle_aged"),
        "age_group_senior":           int(age_group == "senior"),
        "age_group_young":            int(age_group == "young"),
    }
    return pd.DataFrame([row])

# ── Routes ─────────────────────────────────────────────────
@app.get("/")
def root():
    return {
        "message":   "Netflix Churn Prediction API is running!",
        "version":   metadata.get("version", "unknown"),
        "model_auc": metadata.get("auc", "unknown")
    }

@app.get("/health")
def health():
    return {
        "status":     "healthy",
        "model":      "xgboost",
        "version":    metadata.get("version", "unknown"),
        "checked_at": str(datetime.datetime.now())
    }

@app.post("/predict")
def predict(customer: CustomerData):
    try:
        features        = build_features(customer)
        features_scaled = scaler.transform(features)
        churn_prob      = model.predict_proba(features_scaled)[0][1]
        churn_label     = int(churn_prob >= 0.5)

        if churn_prob >= 0.75:
            risk = "HIGH"
        elif churn_prob >= 0.45:
            risk = "MEDIUM"
        else:
            risk = "LOW"

        result = {
            "churn_probability": round(float(churn_prob), 4),
            "will_churn":        bool(churn_label),
            "risk_level":        risk,
            "recommendation": (
                "Immediate retention offer needed!"   if risk == "HIGH"   else
                "Monitor closely and engage customer" if risk == "MEDIUM" else
                "Customer is healthy"
            ),
            "predicted_at": str(datetime.datetime.now())
        }

        logger.info(json.dumps({
            "event":             "prediction",
            "churn_probability": result["churn_probability"],
            "risk_level":        result["risk_level"],
            "predicted_at":      result["predicted_at"]
        }))

        log_entry = {**customer.dict(), **result}
        s3.put_object(
            Bucket=BUCKET_NAME,
            Key=f"logs/predictions/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.json",
            Body=json.dumps(log_entry)
        )

        return result

    except Exception as e:
        logger.error(f"Prediction error: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/model/info")
def model_info():
    return {
        "model_type":    "XGBoost",
        "model_version": metadata.get("version"),
        "auc_score":     metadata.get("auc"),
        "accuracy":      metadata.get("accuracy"),
        "f1_score":      metadata.get("f1_score"),
        "trained_at":    metadata.get("trained_at"),
        "features":      metadata.get("features")
    }

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# Save the file
with open(save_path, 'w') as f:
    f.write(api_code)

print("✅ API file created successfully!")
print(f"📁 Saved to: {save_path}")

✅ API file created successfully!
📁 Saved to: /home/sagemaker-user/churn-mlops/src/serving/api.py


In [2]:
import subprocess
import time
import os

# Correct path for SageMaker
serving_dir = os.path.expanduser("~/churn-mlops/src/serving")

print("🚀 Starting FastAPI server...")

process = subprocess.Popen(
    ["python", "-m", "uvicorn",
     "api:app",
     "--host", "0.0.0.0",
     "--port", "8000"],
    cwd=serving_dir,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ Waiting for server to start...")
time.sleep(10)

print("✅ API server started!")
print("🌐 API running at: http://localhost:8000")
print("📖 API docs at:    http://localhost:8000/docs")

🚀 Starting FastAPI server...
⏳ Waiting for server to start...
✅ API server started!
🌐 API running at: http://localhost:8000
📖 API docs at:    http://localhost:8000/docs


In [3]:
import requests

try:
    response = requests.get("http://localhost:8000/health")
    print("✅ Health check passed!")
    print(f"Response: {response.json()}")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Health check passed!
Response: {'status': 'healthy', 'model': 'xgboost', 'version': '20260413_081716', 'checked_at': '2026-04-13 12:11:21.840081'}


In [4]:
test_customer = {
    "age": 62.0,
    "watch_hours": 1.12,
    "last_login_days": 43.0,
    "monthly_fee": 17.99,
    "number_of_profiles": 4.0,
    "avg_watch_time_per_day": 0.03,
    "gender": "Female",
    "subscription_type": "Premium",
    "region": "Asia",
    "payment_method": "Gift Card"
}

response = requests.post(
    "http://localhost:8000/predict",
    json=test_customer
)

result = response.json()
print("🎯 Prediction Result:")
print(f"   Churn Probability: {result['churn_probability']:.1%}")
print(f"   Will Churn:        {result['will_churn']}")
print(f"   Risk Level:        {result['risk_level']}")
print(f"   Recommendation:    {result['recommendation']}")
print(f"   Predicted At:      {result['predicted_at']}")

🎯 Prediction Result:
   Churn Probability: 99.9%
   Will Churn:        True
   Risk Level:        HIGH
   Recommendation:    Immediate retention offer needed!
   Predicted At:      2026-04-13 12:11:31.992576


In [5]:
response = requests.get("http://localhost:8000/model/info")
info = response.json()

print("📊 Model Information:")
print(f"   Model Type:    {info['model_type']}")
print(f"   Version:       {info['model_version']}")
print(f"   AUC Score:     {info['auc_score']}")
print(f"   Accuracy:      {info['accuracy']}")
print(f"   F1 Score:      {info['f1_score']}")
print(f"   Trained At:    {info['trained_at']}")
print(f"\n✅ API is fully working!")
print(f"\n🎉 Step 5 Complete!")
print(f"   Your model is now a live REST API!")
print(f"   Anyone can send customer data and get predictions!")

📊 Model Information:
   Model Type:    XGBoost
   Version:       20260413_081716
   AUC Score:     0.9966
   Accuracy:      0.9561
   F1 Score:      0.9545
   Trained At:    2026-04-13 08:17:16.959714

✅ API is fully working!

🎉 Step 5 Complete!
   Your model is now a live REST API!
   Anyone can send customer data and get predictions!
